# LABORATORIO N° 02 — Fundamentos de Gestión de Datos

## SQL Básico — Consultas y Filtros

| | |
|:---|:---|
| **Curso** | Fundamentos de Gestión de Datos — C56516 |
| **Semana** | 2 |
| **Tema** | SQL Básico — Consultas y Filtros |
| **Docente** | Pilar Rocío Sayán Mejía |
| **Semestre** | 2026-I |

---
> **Objetivo:** Conectar a SQLite en Google Colab, cargar un dataset sintético de ventas retail
> y ejecutar consultas SQL con SELECT, WHERE, ORDER BY, LIMIT, DISTINCT, COUNT, SUM, AVG, MAX, MIN, GROUP BY y HAVING.


---
# ACTIVIDAD 1: Revisión de Conceptos — SQL Básico

Completa la siguiente tabla con **tus propias palabras**. No copies textualmente del material; demuestra comprensión.

| N° | Concepto | Definición con tus propias palabras |
|:---:|:---|:---|
| 1 | ¿Qué es SQL? | *(Escribe aquí)* |
| 2 | SELECT | *(Escribe aquí)* |
| 3 | WHERE | *(Escribe aquí)* |
| 4 | ORDER BY / LIMIT | *(Escribe aquí)* |
| 5 | DISTINCT | *(Escribe aquí)* |
| 6 | COUNT / SUM / AVG | *(Escribe aquí)* |
| 7 | MAX / MIN | *(Escribe aquí)* |
| 8 | GROUP BY / HAVING | *(Escribe aquí)* |


---
# ACTIVIDAD 2: Desarrollo Práctico — Consultas SQL en SQLite

En esta actividad conectarás una base de datos SQLite desde Google Colab con un dataset
**sintético de ventas retail** y ejecutarás consultas SQL progresivas.

*Ref: Silberschatz, A., Korth, H. F. y Sudarshan, S. (2019). Database System Concepts (7.ª ed.). McGraw-Hill. Capítulos 3–4.*


## Paso 1: Configuración del entorno y generación del dataset sintético en SQLite

**¿Qué hacemos?** Importamos las librerías, generamos 500 registros de ventas retail
con `numpy` y `random` (sin necesidad de internet) y cargamos los datos en SQLite.

*Ref: Documentación oficial de SQLite — https://www.sqlite.org/lang_select.html*


In [ ]:
import sqlite3, numpy as np, random, pandas as pd, matplotlib.pyplot as plt, warnings
from datetime import datetime, timedelta
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

# ── Generar dataset sintético de ventas retail ────────────────────────────────
np.random.seed(42)
random.seed(42)
n = 500

categorias = {
    'Technology':      ['Phones', 'Accessories', 'Machines', 'Tablets', 'Monitors'],
    'Furniture':       ['Chairs', 'Tables', 'Bookcases', 'Furnishings', 'Storage'],
    'Office Supplies': ['Binders', 'Paper', 'Art', 'Labels', 'Fasteners', 'Appliances'],
}
regiones  = ['West', 'East', 'Central', 'South']
segmentos = ['Consumer', 'Corporate', 'Home Office']
modos_env = ['Standard Class', 'Second Class', 'First Class', 'Same Day']
clientes  = [f'Cliente {i}' for i in range(1, 61)]
fecha_ini = datetime(2023, 1, 1)

filas = []
for i in range(n):
    cat    = random.choice(list(categorias.keys()))
    subcat = random.choice(categorias[cat])
    prod   = f'{subcat} {random.choice(["Pro","Plus","Basic","Elite"])} {random.randint(100,999)}'
    fecha  = fecha_ini + timedelta(days=random.randint(0, 730))
    filas.append({
        'Order_ID':      f'US-{fecha.year}-{random.randint(100000,199999)}',
        'Order_Date':    fecha.strftime('%d/%m/%Y'),
        'Ship_Mode':     random.choices(modos_env, weights=[60,20,15,5])[0],
        'Customer_Name': random.choice(clientes),
        'Segment':       random.choices(segmentos, weights=[50,30,20])[0],
        'Region':        random.choices(regiones,  weights=[30,30,25,15])[0],
        'Category':      cat,
        'Sub_Category':  subcat,
        'Product_Name':  prod,
        'Sales':         round(random.uniform(10, 2500), 2),
    })

df = pd.DataFrame(filas)

# ── Cargar en SQLite ──────────────────────────────────────────────────────────
conn = sqlite3.connect('ventas.db')
df.to_sql('ventas', conn, if_exists='replace', index=False)

print('✅ Dataset generado y cargado en SQLite')
print(f'   Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas')
print(f'   Columnas: {df.columns.tolist()}')
print()
df.head(5)


### Pregunta 1
¿Cuántos registros y columnas tiene el dataset? ¿Qué ventaja tiene usar datos sintéticos
generados con `numpy` frente a descargar un archivo de internet?

**Respuesta:**

*Escribe tu respuesta aquí...*


---
## Paso 2: Primera consulta SELECT — Exploración con DISTINCT

**¿Qué hacemos?** Ejecutamos nuestra primera consulta SQL para explorar el dataset
usando `SELECT` con columnas específicas y `DISTINCT` para eliminar duplicados.

*Ref: Silberschatz et al. (2019), Capítulo 3: Introduction to SQL.*


In [ ]:
# ── Consulta 1: SELECT con DISTINCT y ORDER BY ──────────────────────────────
q1 = '''
SELECT DISTINCT Category, Sub_Category, Region
FROM ventas
ORDER BY Category, Sub_Category
LIMIT 15
'''
resultado1 = pd.read_sql_query(q1, conn)
print('CATEGORÍAS Y REGIONES ÚNICAS (primeras 15):')
print(resultado1.to_string(index=False))


In [ ]:
# ── Ver todas las categorías con conteo ──────────────────────────────────────
q_cats = '''
SELECT DISTINCT Category, COUNT(*) AS registros
FROM ventas
GROUP BY Category
ORDER BY registros DESC
'''
cats = pd.read_sql_query(q_cats, conn)
print('RESUMEN POR CATEGORÍA:')
print(cats.to_string(index=False))


### Pregunta 2
¿Para qué sirve `DISTINCT` en esta consulta? ¿Qué diferencia hay entre `SELECT *`
y `SELECT` con columnas específicas? Indica cuándo usarías cada uno.

**Respuesta:**

*Escribe tu respuesta aquí...*


---
## Paso 3: Filtros con WHERE — Consultas específicas de negocio

**¿Qué hacemos?** Usamos la cláusula `WHERE` con múltiples condiciones para filtrar
datos según criterios específicos de negocio.

*Ref: Ramakrishnan, R. y Gehrke, J. (2003). Database Management Systems. Capítulo 5.*


In [ ]:
# ── Consulta 2: Ventas de Technology con monto mayor a 500 ──────────────────
q2 = '''
SELECT Order_Date, Customer_Name, Category,
       Sub_Category, ROUND(Sales, 2) AS Sales
FROM ventas
WHERE Category = 'Technology'
  AND Sales > 500
ORDER BY Sales DESC
LIMIT 15
'''
resultado2 = pd.read_sql_query(q2, conn)
print('VENTAS DE TECHNOLOGY > S/.500:')
print(resultado2.to_string(index=False))


In [ ]:
# ── Ventas en la región West entre 100 y 1000 ────────────────────────────────
q2b = '''
SELECT Region, Segment, Customer_Name,
       Category, ROUND(Sales, 2) AS Sales
FROM ventas
WHERE Region = 'West'
  AND Sales BETWEEN 100 AND 1000
ORDER BY Sales DESC
LIMIT 10
'''
resultado2b = pd.read_sql_query(q2b, conn)
print('VENTAS EN WEST (100 - 1000):')
print(resultado2b.to_string(index=False))


### Pregunta 3
¿Qué condiciones usaste en `WHERE`? ¿Qué resultado obtuviste?
Indica **una conclusión de negocio** (ej.: "El segmento X es el más rentable en Technology").

**Respuesta:**

*Escribe tu respuesta aquí...*


---
## Paso 4: ORDER BY y LIMIT — Ranking de productos

**¿Qué hacemos?** Combinamos `GROUP BY`, `ORDER BY` y `LIMIT` para obtener los
productos más vendidos, con COUNT, SUM, AVG, MAX y MIN.

*Ref: Silberschatz et al. (2019), Capítulo 3: Ordering the Display of Tuples.*


In [ ]:
# ── Consulta 3: Top 10 productos por ventas totales ─────────────────────────
q3 = '''
SELECT Product_Name, Category,
       COUNT(*)                   AS num_pedidos,
       ROUND(SUM(Sales), 2)       AS total_ventas,
       ROUND(AVG(Sales), 2)       AS promedio_venta,
       ROUND(MAX(Sales), 2)       AS venta_maxima,
       ROUND(MIN(Sales), 2)       AS venta_minima
FROM ventas
GROUP BY Product_Name, Category
ORDER BY total_ventas DESC
LIMIT 10
'''
resultado3 = pd.read_sql_query(q3, conn)
print('TOP 10 PRODUCTOS POR VENTAS TOTALES:')
print(resultado3.to_string(index=False))


In [ ]:
# ── Visualización del ranking ─────────────────────────────────────────────────
plt.figure(figsize=(12, 6))
nombres = [n[:35] + '...' if len(n) > 35 else n for n in resultado3['Product_Name']]
plt.barh(nombres[::-1], resultado3['total_ventas'][::-1], color='steelblue', edgecolor='white')
plt.xlabel('Ventas Totales (S/.)')
plt.title('Top 10 Productos por Ventas Totales')
plt.tight_layout()
plt.show()


### Pregunta 4
¿Cuál es el producto con mayor total de ventas? ¿Qué conclusión de negocio extraes
del ranking? ¿Por qué es útil usar `LIMIT` en este tipo de consultas?

**Respuesta:**

*Escribe tu respuesta aquí...*


---
## Paso 5: Funciones de agregación — Estadísticas por categoría

**¿Qué hacemos?** Usamos `COUNT`, `SUM`, `AVG`, `MAX` y `MIN` con `GROUP BY`
para obtener un resumen estadístico completo de las ventas por categoría.

*Ref: Silberschatz et al. (2019), Capítulo 3: Aggregate Functions.*


In [ ]:
# ── Consulta 4: Estadísticas completas por categoría ────────────────────────
q4 = '''
SELECT Category,
       COUNT(DISTINCT Order_ID)   AS total_pedidos,
       ROUND(SUM(Sales), 2)       AS ingresos_totales,
       ROUND(AVG(Sales), 2)       AS ticket_promedio,
       ROUND(MAX(Sales), 2)       AS venta_maxima,
       ROUND(MIN(Sales), 2)       AS venta_minima
FROM ventas
GROUP BY Category
ORDER BY ingresos_totales DESC
'''
resultado4 = pd.read_sql_query(q4, conn)
print('ESTADÍSTICAS POR CATEGORÍA:')
print(resultado4.to_string(index=False))


In [ ]:
# ── Visualización: ingresos totales y ticket promedio por categoría ───────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#1B3A5C', '#2E75B6', '#5BA3D9']

axes[0].bar(resultado4['Category'], resultado4['ingresos_totales'], color=colors)
axes[0].set_title('Ingresos Totales por Categoría')
axes[0].set_ylabel('Ingresos (S/.)')
for i, v in enumerate(resultado4['ingresos_totales']):
    axes[0].text(i, v + 100, f'S/.{v:,.0f}', ha='center', fontsize=9)

axes[1].bar(resultado4['Category'], resultado4['ticket_promedio'], color=colors)
axes[1].set_title('Ticket Promedio por Categoría')
axes[1].set_ylabel('Promedio (S/.)')
for i, v in enumerate(resultado4['ticket_promedio']):
    axes[1].text(i, v + 10, f'S/.{v:,.0f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()


### Pregunta 5
¿Qué categoría genera mayores ingresos totales? ¿Cuál tiene el mayor ticket promedio?
Escribe **dos conclusiones de negocio**.

**Respuesta:**

*Escribe tu respuesta aquí...*


---
## Paso 6: HAVING — Filtrado avanzado por región y segmento

**¿Qué hacemos?** Usamos `HAVING` para filtrar grupos después de la agregación —
diferente a `WHERE` que filtra filas individuales.

*Ref: Silberschatz et al. (2019), Capítulo 3: The Having Clause.*


In [ ]:
# ── Consulta 5: Ventas por región y segmento (solo grupos > 10 000) ──────────
q5 = '''
SELECT Region, Segment,
       COUNT(DISTINCT Order_ID)   AS num_pedidos,
       ROUND(SUM(Sales), 2)       AS total_ventas,
       ROUND(AVG(Sales), 2)       AS promedio_por_pedido
FROM ventas
WHERE Sales > 0
GROUP BY Region, Segment
HAVING SUM(Sales) > 10000
ORDER BY total_ventas DESC
'''
resultado5 = pd.read_sql_query(q5, conn)
print('VENTAS POR REGIÓN Y SEGMENTO (grupos con total > S/.10 000):')
print(resultado5.to_string(index=False))


In [ ]:
# ── Demostración: diferencia entre WHERE y HAVING ────────────────────────────
print('DIFERENCIA ENTRE WHERE Y HAVING')
print('=' * 60)
print()
print('WHERE filtra FILAS antes de agrupar:')
qDemo1 = "SELECT Category, COUNT(*), ROUND(SUM(Sales),2) FROM ventas WHERE Sales > 200 GROUP BY Category"
print(pd.read_sql_query(qDemo1, conn).to_string(index=False))
print()
print('HAVING filtra GRUPOS después de agrupar:')
qDemo2 = "SELECT Category, COUNT(*), ROUND(SUM(Sales),2) AS total FROM ventas GROUP BY Category HAVING total > 50000"
print(pd.read_sql_query(qDemo2, conn).to_string(index=False))


### Pregunta 6
¿Qué hace `HAVING`? ¿En qué se diferencia de `WHERE`?
¿Qué región y segmento tiene las mejores ventas? Indica una conclusión de negocio.

**Respuesta:**

*Escribe tu respuesta aquí...*


---
## Paso 7: Cerrar la conexión SQLite

**Buena práctica:** siempre cerrar la conexión a la base de datos al terminar.


In [ ]:
conn.close()
print('✅ Conexión SQLite cerrada correctamente.')
print()
print('RESUMEN DE CONSULTAS EJECUTADAS:')
print('  Consulta 1: SELECT DISTINCT + ORDER BY  — exploración inicial')
print('  Consulta 2: WHERE con múltiples condiciones — filtros de negocio')
print('  Consulta 3: GROUP BY + COUNT/SUM/AVG/MAX/MIN + LIMIT — ranking')
print('  Consulta 4: Funciones de agregación por categoría')
print('  Consulta 5: HAVING — filtrado post-agrupación por región/segmento')


---
# ACTIVIDAD 3: Caso de Estudio — Análisis de Ventas con SQL

**Contexto:** La gerencia de *DataVentas S.A.* necesita un informe de desempeño.
Responde las siguientes preguntas de negocio **usando SQL** sobre la base de datos SQLite.
Para cada respuesta, escribe la consulta SQL utilizada e interpreta el resultado.


In [ ]:
# Reconectar para la actividad 3
conn = sqlite3.connect('ventas.db')
print('✅ Conexión restablecida')


### Pregunta A
¿Cuáles son los **5 clientes** con mayor monto de compras acumulado?

In [ ]:
qA = '''
-- ESCRIBE O MODIFICA TU CONSULTA SQL AQUÍ
SELECT Customer_Name,
       COUNT(DISTINCT Order_ID) AS num_ordenes,
       ROUND(SUM(Sales), 2)     AS total_compras
FROM ventas
GROUP BY Customer_Name
ORDER BY total_compras DESC
LIMIT 5
'''
resultadoA = pd.read_sql_query(qA, conn)
print(resultadoA.to_string(index=False))


**Conclusión de negocio (Pregunta A):**

*Escribe tu interpretación aquí...*

### Pregunta B
¿Qué **región** concentra más ventas? ¿Hay diferencias significativas entre regiones?

In [ ]:
qB = '''
-- ESCRIBE O MODIFICA TU CONSULTA SQL AQUÍ
SELECT Region,
       COUNT(*) AS num_pedidos,
       ROUND(SUM(Sales), 2) AS total_ventas
FROM ventas
GROUP BY Region
ORDER BY total_ventas DESC
'''
resultadoB = pd.read_sql_query(qB, conn)
print(resultadoB.to_string(index=False))


**Conclusión de negocio (Pregunta B):**

*Escribe tu interpretación aquí...*

### Pregunta C
¿Cuál es el **modo de envío** más utilizado por clientes del segmento **Corporate**?

In [ ]:
qC = '''
-- ESCRIBE O MODIFICA TU CONSULTA SQL AQUÍ
SELECT Ship_Mode,
       COUNT(*) AS frecuencia,
       ROUND(SUM(Sales), 2) AS total_ventas
FROM ventas
WHERE Segment = 'Corporate'
GROUP BY Ship_Mode
ORDER BY frecuencia DESC
'''
resultadoC = pd.read_sql_query(qC, conn)
print(resultadoC.to_string(index=False))


**Conclusión de negocio (Pregunta C):**

*Escribe tu interpretación aquí...*

### Pregunta D
¿Qué diferencia existe entre `WHERE` y `HAVING` en una consulta con `GROUP BY`?
Da un **ejemplo concreto** con el dataset.

In [ ]:
# Ejemplo WHERE vs HAVING con el mismo dataset
print('WHERE filtra FILAS (antes de agrupar):')
qD1 = "SELECT Category, COUNT(*) AS conteo, ROUND(SUM(Sales),2) AS total FROM ventas WHERE Sales > 200 GROUP BY Category"
print(pd.read_sql_query(qD1, conn).to_string(index=False))
print()
print('HAVING filtra GRUPOS (después de agrupar):')
qD2 = "SELECT Category, COUNT(*) AS conteo, ROUND(SUM(Sales),2) AS total FROM ventas GROUP BY Category HAVING total > 50000"
print(pd.read_sql_query(qD2, conn).to_string(index=False))


**Respuesta (Pregunta D):**

*Explica con tus propias palabras la diferencia...*

### Pregunta E
Basándote en los resultados, ¿qué **3 acciones estratégicas** recomendarías a la gerencia?
Justifica con datos concretos obtenidos en este laboratorio.

In [ ]:
conn.close()
print('✅ Caso de estudio completado. Conexión cerrada.')


**Recomendaciones estratégicas:**

**Acción 1:** *...*

**Acción 2:** *...*

**Acción 3:** *...*


---
# Conclusiones

**1.** *Escribe tu primera conclusión técnica aquí...*

**2.** *Escribe tu segunda conclusión aquí...*

**3.** *Escribe tu tercera conclusión aquí...*

---
© 2026 — Fundamentos de Gestión de Datos — TECSUP | Docente: Pilar Rocío Sayán Mejía
